# AllSides media-bias types, scored by Jev on 3,000 texts

750 random stories from the `test` split × 4 texts each: the left, centre and right source
articles (labelled by slot) and the LLM neutral summary (unlabelled). 3,000 texts in total.

Each text received **one request carrying 18 questions**, answered independently:
- **12 presence checks** (yes/no probability), one per AllSides type observable in a single text:
  spin, sensationalism, loaded words, subjective adjectives, opinion as fact, unsubstantiated
  claims, unattributed sources, mudslinging, mind reading, negativity, one-sidedness, flawed logic.
- **5 direction questions** (Choice): whose vocabulary; which side the slant favours; whose
  viewpoint comes first; which side is attacked; elite vs populist.
- **`outlet_style`**, the ideology question from the earlier Jev run, for comparison.

Not covered, because a single text can't show them: story choice, story placement, photo bias, and
omission (which needs the three articles compared side by side).

In [ ]:
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix
J = Path.cwd().resolve()  # jev/; run the notebook from its own folder
R = [json.loads(l) for l in open(J / "results/bias_types.jsonl")]
PRES = ["spin", "sensationalism", "loaded_words", "subjective_adjectives", "opinion_as_fact", "unsubstantiated",
        "unattributed_sources", "mudslinging", "mind_reading", "negativity", "one_sided", "flawed_logic"]
DIR = {"word_choice_side": ["left", "right", "neither"], "slant_side": ["left", "right", "neither"],
       "viewpoint_first_side": ["left", "right", "neither"], "attack_target": ["left", "right", "none"],
       "elite_populist": ["elite", "populist", "neither"], "outlet_style": ["left", "centre", "right"]}
rows = []
for r in R:
    d = {k: r[k] for k in ("id", "story_id", "slot", "label", "publisher", "input_tokens")}
    d.update({q: r[q]["p"] for q in PRES})
    for q, opts in DIR.items():
        for o in opts: d[f"{q}:{o}"] = r[q]["p"].get(o, 0.0)
    rows.append(d)
df = pd.DataFrame(rows)
ORDER = ["left", "center", "right", "neutral"]
spend = json.loads((J / "results/spend.json").read_text())
print(f"{len(df)} texts, {df.story_id.nunique()} stories;", df.slot.value_counts().to_dict())
print(f"mean {df.input_tokens.mean():.0f} input tokens per text (18 questions);"
      f" this run ≈ ${df.input_tokens.sum() * 0.042 / 1e6:.3f}; ledger total ${spend['usd']}")

## 1. How often each bias type appears, by text type

Mean yes-probability. Higher means more of that bias.

In [ ]:
pres = df.groupby("slot")[PRES].mean().reindex(ORDER).T
pres["partisan_mean"] = pres[["left", "right"]].mean(1)
pres["neutral_vs_centre"] = pres["neutral"] - pres["center"]
pres.round(3).sort_values("neutral_vs_centre")

In [ ]:
ax = pres[ORDER].plot.barh(figsize=(9, 8), width=0.8)
ax.set(xlabel="mean P(yes)", title="AllSides bias types by text type (750 stories)"); ax.invert_yaxis()
plt.tight_layout(); plt.show()

In [ ]:
# overall bias load: mean over the 12 presence checks
df["bias_load"] = df[PRES].mean(1)
load = df.groupby("slot").bias_load.describe().reindex(ORDER)[["mean", "25%", "50%", "75%"]]
# paired comparison within each story: is the neutral summary less biased than its own sources?
w = df.pivot(index="story_id", columns="slot", values="bias_load")
for s in ["left", "center", "right"]:
    load.loc[s, "neutral_lower_in_%_of_stories"] = (w.neutral < w[s]).mean() * 100
load.round(3)

## 2. Direction questions against the true label (labelled articles only)

In [ ]:
lab = df[df.slot != "neutral"].copy(); lab["label"] = lab.label.astype(int)
out = []
for q, opts in DIR.items():
    tab = lab.groupby("slot")[[f"{q}:{o}" for o in opts]].mean().reindex(["left", "center", "right"])
    tab.columns = opts
    out.append(tab.assign(question=q).set_index("question", append=True))
pd.concat(out).round(3)

In [ ]:
# each directional question used as a stand-alone 3-way classifier
def as_pred(q):
    if q == "outlet_style":
        P = lab[[f"{q}:left", f"{q}:centre", f"{q}:right"]].values
    else:
        o = DIR[q]; P = lab[[f"{q}:{o[0]}", f"{q}:{o[2]}", f"{q}:{o[1]}"]].values   # left, neither->centre, right
    return P.argmax(1), P
y = lab.label.values; res = []
for q in ["outlet_style", "word_choice_side", "slant_side", "viewpoint_first_side", "attack_target"]:
    pred, P = as_pred(q)
    res.append({"question": q, "accuracy": (pred == y).mean(), "macro_f1": f1_score(y, pred, average="macro"),
                "recall_L": (pred[y == 0] == 0).mean(), "recall_C": (pred[y == 1] == 1).mean(),
                "recall_R": (pred[y == 2] == 2).mean()})
pd.DataFrame(res).round(4)

## 3. All 17 bias signals combined

A logistic regression over every presence and direction probability, with story-grouped 5-fold
cross-validation so no story's articles appear in both train and test. This measures how much
ideology signal the AllSides features carry together.

In [ ]:
from sklearn.model_selection import GroupKFold
feat_all = PRES + [c for c in df.columns if ":" in c and not c.startswith("outlet_style")]
def cv(feats):
    X = lab[feats].values
    clf = LogisticRegression(max_iter=2000, C=1.0)
    P = cross_val_predict(clf, X, y, cv=GroupKFold(5), groups=lab.story_id, method="predict_proba")
    p = P.argmax(1)
    return {"accuracy": (p == y).mean(), "macro_f1": f1_score(y, p, average="macro"),
            "auc_macro": roc_auc_score(y, P, multi_class="ovr", average="macro")}, p
rows = []
for name, feats in [("presence only (12)", PRES),
                    ("direction only", [c for c in feat_all if ":" in c]),
                    ("all bias signals (27 features)", feat_all),
                    ("all + outlet_style", feat_all + ["outlet_style:left", "outlet_style:centre", "outlet_style:right"])]:
    m, p = cv(feats); rows.append({"features": name, **m})
    if name.startswith("all +"): best_pred = p
pd.DataFrame(rows).round(4)

In [ ]:
cm = confusion_matrix(y, best_pred, labels=[0, 1, 2])
print("confusion matrix, all signals + outlet_style (cross-validated):")
pd.DataFrame(cm, index=["true Left", "true Centre", "true Right"], columns=["pred Left", "pred Centre", "pred Right"])

In [ ]:
# which signals separate the classes? one-vs-rest AUC per feature
auc = {}
for f in feat_all:
    auc[f] = {c: roc_auc_score((y == i).astype(int), lab[f]) for i, c in enumerate(["vs_Left", "vs_Centre", "vs_Right"])}
a = pd.DataFrame(auc).T
a["max_dev_from_0.5"] = (a - 0.5).abs().max(1)
a.sort_values("max_dev_from_0.5", ascending=False).head(12).round(3)

## 4. Neutral summaries: direction and ideology

In [ ]:
neu = df[df.slot == "neutral"]
out = {}
for q in ["outlet_style", "word_choice_side", "slant_side", "viewpoint_first_side", "attack_target"]:
    opts = DIR[q]
    out[q] = {o: neu[f"{q}:{o}"].mean() for o in opts}
pd.DataFrame(out).T.round(3)

## Notes

- Presence probabilities are Jev's answers to one question per type, not calibrated rates. Compare
  them across text types; the absolute values are less meaningful.
- The labels are outlet-level. Where a direction question disagrees with the label, the article
  may genuinely be milder than its outlet.
- Omission bias isn't here. It needs the three articles of a story compared in one request.
- Raw answers (full distributions): `jev/results/bias_types.jsonl`.